In [5]:
from sklearn.ensemble import GradientBoostingClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, fbeta_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from joblib import dump
import numpy as np
pd.set_option("display.float_format", lambda x: "%0.3f" % x)
np.set_printoptions(suppress=True)

In [2]:
data = pd.read_csv("Data/processed_data")
data

,home_ownership,purpose,annual_income,int_rate,loan_amount,is_loss,DTI_USD,annual_income_ru,loan_ammount_ru,int_rate_ru
0,RENT,car,30000.000,0.153,2500,1,0.083,444000.000,37000.000,0.219
1,RENT,car,48000.000,0.186,3000,0,0.062,710400.000,44400.000,0.267
2,RENT,car,50000.000,0.160,12000,1,0.240,740000.000,177600.000,0.229
3,MORTGAGE,car,42000.000,0.106,4500,0,0.107,621600.000,66600.000,0.153
4,MORTGAGE,car,83000.000,0.060,3500,0,0.042,1228400.000,51800.000,0.086
...,...,...,...,...,...,...,...,...,...,...
38568,MORTGAGE,other,100000.000,0.130,24250,0,0.242,1480000.000,358900.000,0.186
38569,RENT,other,50000.000,0.135,25200,0,0.504,740000.000,372960.000,0.193
38570,RENT,other,65000.000,0.175,25000,0,0.385,962000.000,370000.000,0.251
38571,RENT,other,368000.000,0.182,24000,0,0.065,5446400.000,355200.000,0.262


In [3]:
X = data[["annual_income_ru", "loan_ammount_ru", "int_rate_ru", "home_ownership", "purpose"]] 
Y = data["is_loss"]

test_frac = 0.15
validation_frac = round(test_frac / (1 - test_frac), ndigits=2) #0.18
X_temp, X_test, Y_temp, Y_test = train_test_split(
X, Y, test_size=test_frac, random_state=42, stratify=Y
)

# Разделяем оставшиеся данные на обучающую и валидационную выборки
X_train, X_val, Y_train, Y_val = train_test_split(
X_temp, Y_temp, test_size=validation_frac, random_state=42, stratify=Y_temp
)

Y_train = pd.DataFrame(Y_train)
Y_test = pd.DataFrame(Y_test)
print(f"X_train: {X_train.shape}\nX_test: {X_test.shape}\nY_train: {Y_train.shape}\nY_test: {Y_test.shape}")

X_train: (26885, 5)
X_test: (5786, 5)
Y_train: (26885, 1)
Y_test: (5786, 1)


In [7]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, [0,1,2]),
        ('cat', categorical_transformer, [3,4])
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        subsample=0.8,
        min_samples_split=10,
        min_samples_leaf=4
    ))])

pipeline.fit(X_train, Y_train)
prediction = pipeline.predict(X_test)
recall = recall_score(Y_test, prediction)
precision = precision_score(Y_test, prediction)
print(f"Recall: {recall}\nPrecision:{precision}")

c:\Github\Credit-s-RIsk-Prediction\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Recall: 0.01125
Precision:0.4090909090909091


In [8]:
model_path = "models/Boosting.joblib"
dump(pipeline, model_path)

['models/Boosting.joblib']